# Para practicar · Los logs de diciembre

En diciembre, el equipo del chatbot de Folio, la librería de la demo de clase, ha probado una versión nueva del prompt, la `1.20`, más corta: gasta menos tokens y por tanto sale más barata. La pregunta es si **compensa**: ¿acierta lo suficiente comparada con la `1.10`?

Es el mismo recorrido que la demo de clase, con dos cambios: los datos son de diciembre y soporte ya no manda un CSV, sino directamente un **Excel**.

| Fichero | Formato |
| --- | --- |
| `logs_diciembre.jsonl` | JSONL, con la respuesta del modelo anidada |
| `valoraciones_diciembre.xlsx` | Excel |

**Cómo funciona.** Cada ejercicio tiene huecos marcados con `None` o con `___`. Cambiadlos por lo que pide el comentario y ejecutad la celda de comprobación de debajo. Si os atascáis, la demo de clase hace lo mismo paso a paso.

**Antes de empezar**, instalad en vuestro entorno lo que hace falta: `uv add pyarrow duckdb openpyxl`. Si el notebook ya estaba abierto, reiniciad el kernel después.

In [ ]:
import json
from pathlib import Path

import duckdb
import pandas as pd


def comprobar(nombre, valor, esperado):
    """Compara vuestro resultado con el esperado. No hace falta tocarla."""
    if valor is None:
        print(f"--      {nombre}: sin completar")
    elif valor == esperado:
        print(f"OK      {nombre}")
    else:
        print(f"REVISAD {nombre}: da {valor} y se esperaba {esperado}")

**Preparar los datos.** Esta celda simula lo que os mandarían los equipos de sistemas y de soporte: crea los ficheros en la misma carpeta que el notebook. Ejecutadla una vez y seguid; no hace falta entenderla.

In [ ]:
import json
import random

import pandas as pd

random.seed(1232)
CATEGORIAS = {
    "Pedidos": "¿Dónde está mi pedido?",
    "Devoluciones": "¿Cómo devuelvo un libro?",
    "Pagos": "¿Puedo pagar con Bizum?",
    "Envíos": "¿Cuánto tarda el envío?",
}
ACIERTO = {"1.10": 0.84, "1.20": 0.74}          # probabilidad de acierto de cada versión del prompt

conversaciones, valoraciones = [], []
for i in range(1, 160 + 1):
    dia = random.randint(1, 30)
    version = "1.10" if dia <= 15 else "1.20"
    categoria = random.choice(list(CATEGORIAS))
    correcta = int(random.random() < ACIERTO[version])
    conversaciones.append({
        "id_conversacion": f"dic-{i:04d}",
        "fecha": f"2026-12-{dia:02d}",
        "categoria": categoria,
        "pregunta": CATEGORIAS[categoria],
        "respuesta": {
            "modelo": random.choice(["modelo_a", "modelo_b"]),
            "version_prompt": version,
            "tokens": random.randint(250, 500) if version == "1.10" else random.randint(150, 350),
            "latencia_ms": random.randint(600, 2200),
        },
    })
    nota = random.choice([4, 4.5, 5]) if correcta else random.choice([1, 1.5, 2, 2.5, 3])
    valoraciones.append({"id_conversacion": f"dic-{i:04d}",
                         "revisor": random.choice(["Lucía", "Marcos", "Irene"]),
                         "correcta": correcta, "puntuacion": nota})

with open("logs_diciembre.jsonl", "w", encoding="utf-8") as f:
    for c in conversaciones:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
# este mes, soporte manda directamente el Excel
pd.DataFrame(valoraciones).to_excel("valoraciones_diciembre.xlsx", index=False)
print("Datos preparados:", "logs_diciembre.jsonl", "y", "valoraciones_diciembre.xlsx")

## 1. Leer y aplanar los logs

La lectura línea a línea ya está hecha. Aplanad los registros con `pd.json_normalize`, usando `sep="_"`.

In [ ]:
with open("logs_diciembre.jsonl", encoding="utf-8") as f:
    registros = [json.loads(linea) for linea in f]

logs = None   # Pista: pd.json_normalize(registros, sep="_")

logs

In [ ]:
comprobar("filas y columnas", None if logs is None else logs.shape, (160, 8))
comprobar("hay columna de versión", None if logs is None else "respuesta_version_prompt" in logs.columns, True)

## 2. Leer el Excel de soporte

`pd.read_excel` lee un `.xlsx`. Le pasamos `engine="openpyxl"` para indicar qué librería usa por dentro; sin él, pandas 3 muestra un aviso. En un Excel no hace falta `sep` ni `decimal`: los números ya van guardados como números.

In [ ]:
valoraciones = None   # Pista: pd.read_excel("valoraciones_diciembre.xlsx", engine="openpyxl")

valoraciones

In [ ]:
comprobar("filas y columnas", None if valoraciones is None else valoraciones.shape, (160, 4))
comprobar("puntuación es un número", None if valoraciones is None else str(valoraciones["puntuacion"].dtype), "float64")

## 3. Unir

Pegad a cada conversación su valoración, por `id_conversacion`, conservando todas las conversaciones.

In [ ]:
datos = None   # Pista: logs.merge(valoraciones, on=..., how="left")

datos

In [ ]:
comprobar("filas y columnas", None if datos is None else datos.shape, (160, 11))
comprobar("conversaciones sin valorar", None if datos is None else int(datos["correcta"].isna().sum()), 0)

## 4. Guardar en Parquet

Guardad `datos` en `logs_diciembre.parquet` (sin el índice) y volved a leerlo en `de_parquet`.

In [ ]:
# Pista: datos.to_parquet("logs_diciembre.parquet", index=False)

de_parquet = None   # Pista: pd.read_parquet(...)

de_parquet

In [ ]:
versiones = None if de_parquet is None else sorted(de_parquet["respuesta_version_prompt"].unique().tolist())
comprobar("versiones intactas", versiones, ["1.10", "1.20"])

**Para pensar.** Si lo hubierais guardado en CSV, ¿qué valores tendría la columna de versión al volver a leerlo? ¿Por qué sería un problema para la pregunta de este notebook?

*Escribid aquí vuestra respuesta.*

## 5. Preguntar con DuckDB

Completad la consulta: en `FROM`, el fichero Parquet entre comillas **simples**; en `GROUP BY`, la columna por la que agrupamos (podéis usar el nombre nuevo, `version`). Después, cambiad `None` por `duckdb.sql(consulta).df()`.

In [ ]:
consulta = """
SELECT respuesta_version_prompt AS version,
       round(avg(correcta), 2) AS acierto,
       round(avg(respuesta_tokens)) AS tokens_medios,
       count(*) AS conversaciones
FROM ___
GROUP BY ___
ORDER BY version
"""

resultado = None   # Pista: duckdb.sql(consulta).df()

resultado

In [ ]:
comprobar("acierto de la 1.10", None if resultado is None else float(resultado["acierto"][0]), 0.82)
comprobar("acierto de la 1.20", None if resultado is None else float(resultado["acierto"][1]), 0.73)
comprobar("tokens medios de la 1.20", None if resultado is None else float(resultado["tokens_medios"][1]), 242.0)

## 6. Decidir

Ahora la parte complicada, ¿con qué vesion os quedáis? Justificadlo con los números de la tabla: cuánto se ahorra en tokens, cuánto se pierde en acierto y con cuántas conversaciones lo estáis midiendo. Abrimos debate

*Escribid aquí vuestra respuesta.*

## Para cerrar

Si habéis usado IA en algún paso: anotad qué le preguntasteis, qué os devolvió y qué tuvisteis que corregir o comprobar.

*Escribid aquí vuestra respuesta:*